# Deepfake Video Classification with CNN Frame Features + LSTM

This notebook trains a two-step PyTorch video classifier:

1. Sample a fixed number of frames from each video.
2. Extract a feature vector from each frame with a CNN.
3. Aggregate the frame features with an LSTM.
4. Predict whether the video is fake/deepfake (`1`) or real/not-AI (`0`).

Dataset: https://www.kaggle.com/datasets/farhanabidtech786/deepfake-video-9-6k

The split happens before any dataset-derived preprocessing. The frame normalization statistics are computed from the training split only and then reused unchanged for validation/test data.

## Setup

Install these packages if needed:

```bash
pip install torch torchvision opencv-python scikit-learn tqdm pandas kaggle
```

To use the Kaggle API, create a Kaggle API token and place `kaggle.json` at `~/.kaggle/kaggle.json`, or set `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables. Then set `USE_KAGGLE_API_DOWNLOAD = True` below. The code expects videos to live under class-named folders such as `real`, `fake`, `DeepFake`, or similar. If the dataset folder names differ, update `LABEL_KEYWORDS` in the next cell.

In [ ]:
from pathlib import Path
import random
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
KAGGLE_DATASET = "farhanabidtech786/deepfake-video-9-6k"
USE_KAGGLE_API_DOWNLOAD = True

# Dataset directory. If USE_KAGGLE_API_DOWNLOAD is True, the dataset is downloaded and unzipped here.
# Kaggle-hosted notebook example: DATA_ROOT = Path("/kaggle/input/deepfake-video-9-6k")
DATA_ROOT = Path("data/deepfake-video-9-6k")

if USE_KAGGLE_API_DOWNLOAD:
    import os
    from kaggle.api.kaggle_api_extended import KaggleApi

    kaggle_dir = Path(os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle"))
    for credential_file in (kaggle_dir / "access_token", kaggle_dir / "kaggle.json"):
        if credential_file.exists() and not os.access(credential_file, os.R_OK):
            raise PermissionError(
                f"Kaggle credential file is not readable: {credential_file}. "
                "Fix ownership/permissions, for example: "
                f"sudo chown $USER:$USER {credential_file} && chmod 600 {credential_file}"
            )

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(KAGGLE_DATASET, path=str(DATA_ROOT), unzip=True, quiet=False)
    print(f"Downloaded and extracted {KAGGLE_DATASET} to {DATA_ROOT.resolve()}")

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

# Fake/deepfake videos are label 1. Real/not-AI videos are label 0.
# Matching is case-insensitive and checks path parts relative to DATA_ROOT.
LABEL_KEYWORDS: Dict[int, Tuple[str, ...]] = {
    1: ("fake", "deepfake", "synthetic", "manipulated"),
    0: ("real", "original", "authentic", "not_ai", "not-ai"),
}

def infer_label(path: Path, data_root: Path) -> int:
    relative_path = path.relative_to(data_root)
    parts = [part.lower() for part in relative_path.parts]
    matches = []
    for label, keywords in LABEL_KEYWORDS.items():
        if any(keyword in part for part in parts for keyword in keywords):
            matches.append(label)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous label for path: {path}. Update LABEL_KEYWORDS or build a metadata CSV.")
    raise ValueError(f"Could not infer label from path: {path}. Update LABEL_KEYWORDS or build a metadata CSV.")

def collect_videos(data_root: Path) -> pd.DataFrame:
    if not data_root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root.resolve()}")
    rows = []
    for video_path in sorted(data_root.rglob("*")):
        if video_path.suffix.lower() in VIDEO_EXTENSIONS:
            rows.append({"path": str(video_path), "label": infer_label(video_path, data_root)})
    if not rows:
        raise RuntimeError(f"No videos found under {data_root.resolve()}")
    return pd.DataFrame(rows)

all_videos = collect_videos(DATA_ROOT)
print(all_videos["label"].value_counts().rename({0: "real", 1: "fake"}))
all_videos.head()

## Split before preprocessing

We split video paths and labels before computing any dataset-derived transformation. This avoids leaking test-set information into training.

In [ ]:
TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAIN = 0.10

train_df, test_df = train_test_split(
    all_videos,
    test_size=TEST_SIZE,
    stratify=all_videos["label"],
    random_state=SEED,
)

train_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE_FROM_TRAIN,
    stratify=train_df["label"],
    random_state=SEED,
)

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")
print("\nTrain label counts:\n", train_df["label"].value_counts())
print("\nVal label counts:\n", val_df["label"].value_counts())
print("\nTest label counts:\n", test_df["label"].value_counts())

## Video frame sampling and train-only normalization

The CNN benefits from inputs on a stable numeric scale, so we normalize frames. The mean/std below are estimated from sampled training frames only, then reused for validation and test videos.

In [ ]:
NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
USE_PRETRAINED_CNN = True  # Set False if pretrained ResNet weights cannot be downloaded or are unavailable.

def sample_frame_indices(total_frames: int, num_frames: int) -> np.ndarray:
    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int64)
    if total_frames >= num_frames:
        return np.linspace(0, total_frames - 1, num_frames).round().astype(np.int64)
    indices = np.arange(total_frames, dtype=np.int64)
    pad = np.full(num_frames - total_frames, total_frames - 1, dtype=np.int64)
    return np.concatenate([indices, pad])

def read_video_frames(video_path: str, num_frames: int = NUM_FRAMES, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = sample_frame_indices(total_frames, num_frames)
    frames: List[np.ndarray] = []

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = cap.read()
        if not ok:
            if frames:
                frames.append(frames[-1].copy())
                continue
            frame_rgb = np.zeros((image_size, image_size, 3), dtype=np.uint8)
        else:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_rgb = cv2.resize(frame_rgb, (image_size, image_size), interpolation=cv2.INTER_AREA)
        frames.append(frame_rgb)

    cap.release()

    frame_array = np.stack(frames).astype(np.float32) / 255.0
    return torch.from_numpy(frame_array).permute(0, 3, 1, 2)  # T, C, H, W

def compute_train_mean_std(train_paths: List[str], max_videos: int = 256) -> Tuple[torch.Tensor, torch.Tensor]:
    sampled_paths = train_paths if len(train_paths) <= max_videos else random.sample(train_paths, max_videos)
    channel_sum = torch.zeros(3)
    channel_sq_sum = torch.zeros(3)
    pixel_count = 0

    for path in tqdm(sampled_paths, desc="Estimating train mean/std"):
        frames = read_video_frames(path)  # T, C, H, W in [0, 1]
        channel_sum += frames.sum(dim=(0, 2, 3))
        channel_sq_sum += (frames ** 2).sum(dim=(0, 2, 3))
        pixel_count += frames.shape[0] * frames.shape[2] * frames.shape[3]

    mean = channel_sum / pixel_count
    std = (channel_sq_sum / pixel_count - mean ** 2).sqrt().clamp_min(1e-6)
    return mean, std

train_mean, train_std = compute_train_mean_std(train_df["path"].tolist())
print("train_mean:", train_mean)
print("train_std: ", train_std)

In [ ]:
class DeepfakeVideoDataset(Dataset):
    def __init__(self, frame_df: pd.DataFrame, mean: torch.Tensor, std: torch.Tensor):
        self.frame_df = frame_df.reset_index(drop=True)
        self.mean = mean.view(1, 3, 1, 1)
        self.std = std.view(1, 3, 1, 1)

    def __len__(self) -> int:
        return len(self.frame_df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.frame_df.iloc[idx]
        frames = read_video_frames(row.path)
        frames = (frames - self.mean) / self.std
        label = torch.tensor(float(row.label), dtype=torch.float32)
        return frames, label

train_dataset = DeepfakeVideoDataset(train_df, train_mean, train_std)
val_dataset = DeepfakeVideoDataset(val_df, train_mean, train_std)
test_dataset = DeepfakeVideoDataset(test_df, train_mean, train_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

## Model

The CNN maps each frame to a feature vector. The LSTM receives one feature vector per sampled frame and aggregates temporal evidence across the video. The classifier uses the final LSTM representation for binary prediction.

In [ ]:
class CNNLSTMDeepfakeDetector(nn.Module):
    def __init__(self, feature_dim: int = 512, hidden_dim: int = 256, num_layers: int = 1, dropout: float = 0.3):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if USE_PRETRAINED_CNN else None
        resnet = models.resnet18(weights=weights)
        cnn_feature_dim = resnet.fc.in_features
        resnet.fc = nn.Identity()
        self.cnn = resnet

        self.feature_projection = nn.Sequential(
            nn.Linear(cnn_feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        # frames: B, T, C, H, W
        batch_size, timesteps, channels, height, width = frames.shape
        flat_frames = frames.view(batch_size * timesteps, channels, height, width)
        frame_features = self.cnn(flat_frames)
        frame_features = self.feature_projection(frame_features)
        frame_features = frame_features.view(batch_size, timesteps, -1)
        lstm_out, _ = self.lstm(frame_features)
        video_features = lstm_out[:, -1, :]
        logits = self.classifier(video_features).squeeze(1)
        return logits

model = CNNLSTMDeepfakeDetector().to(device)
sum_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {sum_params:,} total; {trainable_params:,} trainable")

## Training loop

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_epoch(model: nn.Module, loader: DataLoader, training: bool) -> Dict[str, float]:
    model.train(training)
    total_loss = 0.0
    all_labels: List[int] = []
    all_preds: List[int] = []

    progress = tqdm(loader, desc="train" if training else "eval", leave=False)
    for frames, labels in progress:
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(frames)
                loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        probs = torch.sigmoid(logits).detach()
        preds = (probs >= 0.5).long().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.long().cpu().tolist())
        progress.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    return {"loss": avg_loss, "accuracy": accuracy}

best_val_loss = float("inf")
best_model_path = Path("best_cnn_lstm_deepfake_detector.pt")
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
    }
    history.append(row)
    print(row)

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "train_mean": train_mean,
                "train_std": train_std,
                "num_frames": NUM_FRAMES,
                "image_size": IMAGE_SIZE,
                "label_mapping": {"real": 0, "fake": 1},
                "history": history,
            },
            best_model_path,
        )
        print(f"Saved new best model to {best_model_path}")

pd.DataFrame(history)

## Test evaluation

The test loader uses the same frame sampling, resizing, and train-derived normalization as the training and validation loaders. No test-set statistics are used.

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_loss = 0.0
test_labels: List[int] = []
test_preds: List[int] = []
test_probs: List[float] = []

with torch.no_grad():
    for frames, labels in tqdm(test_loader, desc="test"):
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(frames)
        loss = criterion(logits, labels)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        test_loss += loss.item() * labels.size(0)
        test_labels.extend(labels.long().cpu().tolist())
        test_preds.extend(preds.cpu().tolist())
        test_probs.extend(probs.cpu().tolist())

test_loss /= len(test_loader.dataset)
test_acc = accuracy_score(test_labels, test_preds)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print("\nConfusion matrix [real, fake]:")
print(confusion_matrix(test_labels, test_preds))
print("\nClassification report:")
print(classification_report(test_labels, test_preds, target_names=["real", "fake"]))

## Single-video inference

In [ ]:
def predict_video(video_path: str, threshold: float = 0.5) -> Dict[str, float]:
    checkpoint = torch.load(best_model_path, map_location=device)
    frames = read_video_frames(video_path, checkpoint["num_frames"], checkpoint["image_size"])
    mean = checkpoint["train_mean"].view(1, 3, 1, 1)
    std = checkpoint["train_std"].view(1, 3, 1, 1)
    frames = ((frames - mean) / std).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        prob_fake = torch.sigmoid(model(frames)).item()

    return {
        "prob_fake": prob_fake,
        "prediction": "fake" if prob_fake >= threshold else "real",
    }

# Example:
# predict_video("path/to/video.mp4")